In [ ]:
# SecureShare Auth Lifecycle Diagrams (Text)

## 1. Registration Flow

Step-by-step (synchronous CLI path):

1. **User action (CLI UI)**
   - `auth/pages/register.py` → `register_page(cli)`
   - Prompts: `Email`, `Display Name`, `Password`, `Confirm Password`
   - Calls: `cli.auth.register(email, password, display_name)`

2. **Auth.signup + auto-login** (`Auth.register`)
   - Calls Supabase:
     - `client.auth.sign_up({"email": email, "password": password})`
   - If `response.session` **is missing**:
     - Immediately calls `client.auth.sign_in_with_password({"email": email, "password": password})`
     - Uses that `user` + `session` as the source of truth.

3. **Session object creation**
   - Builds in-memory session dict:
     - `user_id` ← `user_data.id`
     - `email` ← `user_data.email`
     - `access_token` ← `session_data.access_token`
     - `refresh_token` ← `session_data.refresh_token`
     - `expires_at` ← `time.time() + session_data.expires_in`
     - `original_lifetime` ← `session_data.expires_in`

4. **Persist and bind to client**
   - `self._save_session(self.session)` →
     - `auth.storage.save_session(session, encrypt=ENABLE_SESSION_ENCRYPTION)`
     - **Development**: writes JSON (or encrypted) to `~/.secure_share/session.json` / `.enc`.
     - **Production**: upserts row into `user_sessions` table via `DatabaseSessionStorage`.
   - `self._set_client_session()` →
     - `ClientManager.set_session(access_token, refresh_token)` → `supabase_client.auth.set_session(...)`.
     - From this point, all Supabase calls carry the user JWT; RLS sees `auth.uid()`.

5. **App profile creation (`public.users`)**
   - `self.client.table("users").insert({ id, email, display_name, cloud_connected=False, created_at=... }).execute()`
   - Uses the **anon key** + current JWT.
   - RLS policy: `"Users can insert own profile"` + GRANTs from `003_user_grant_creation.sql` allow this.

Result:
- Supabase `auth.users` row + `public.users` row.
- Local session saved (file or DB), client bound to tokens, ready for further requests.

---

## 2. Login Flow

1. **User action (CLI UI)**
   - Main menu (`cli/main_menu.py`) → `cli.login()` → `auth/pages/login.py` (`login_page(cli)`).
   - Prompts: `Email`, `Password`.
   - Calls: `cli.auth.login(email, password)`.

2. **Supabase sign-in** (`Auth.login`)
   - `response = client.auth.sign_in_with_password({"email": email, "password": password})`
   - Extracts `response.user` and `response.session`.

3. **Session creation & persistence**
   - Builds same dict as registration (user_id, email, access_token, refresh_token, expires_at, original_lifetime).
   - `self._save_session(self.session)` → file or `user_sessions` table.
   - `self._set_client_session()` → binds JWT to Supabase client.

4. **Profile fetch & cache**
   - `_fetch_user_profile()`:
     - `client.table('users').select('*').eq('id', user_id).execute()`
     - If a row exists: `self.session['profile'] = row` and session is re-saved.

Result:
- User is authenticated for the CLI session; subsequent calls see a live JWT and cached profile.

---

## 3. Normal Authenticated Request Flow

Example: **Account menu** (`cli/account_menu.py`).

1. **Ensure session** (optional but recommended)
   - Call `cli.auth.ensure_authenticated()` or `cli.auth.is_authenticated()`.
   - Internally:
     - `SessionValidator.is_authenticated(session, ensure_valid_callback, refresh_callback)`.
       - Calls `ensure_valid_callback()` → `TokenManager.ensure_valid_session(...)`:
         - If token **expired** or ≥ 80% lifetime used → triggers refresh (see next section).
       - Then calls `client.auth.get_user(access_token)` to confirm token validity.

2. **Perform business query**
   - Account menu uses:
     - `user_id = cli.auth.get_user_id()` (reads from in-memory session).
     - `cli.client.from_('users').select('*').eq('id', user_id).single().execute()`.
   - Supabase sees HTTP request with `Authorization: Bearer <access_token>`.
   - RLS uses `auth.uid()` from the JWT plus policies to authorize.

3. **Error handling (optional)**
   - If a query fails with an auth error (401, expired, etc.):
     - Call `cli.auth.handle_auth_error(error)`.
     - `AuthErrorHandler.handle_auth_error`:
       - Detects auth-related keywords in error.
       - Calls `Auth._refresh_token()` → token refresh flow.
       - Returns `True` if refresh worked → caller retries the original query once.

---

## 4. Token Refresh Flow

Token refresh can be triggered proactively (before expiry) or reactively (on errors).

### 4.1 Proactive / expiry-based refresh

1. **Trigger**
   - Any code path calling `Auth.ensure_authenticated()` or `Auth.is_authenticated()` leads to:
     - `TokenManager.ensure_valid_session(session, refresh_callback)`.

2. **Decision logic** (`TokenManager.ensure_valid_session`)
   - If `expiry_time < now` → token **expired** → call `refresh_callback()`.
   - Else if `original_lifetime` is known:
     - Compute:
       - `elapsed = original_lifetime - time_until_expiry`
       - `threshold = original_lifetime * REFRESH_THRESHOLD` (80%).
     - If `elapsed >= threshold` → proactively refresh.
   - Else (no `original_lifetime`): refresh when less than 12 minutes remain.

3. **Actual refresh** (`Auth._refresh_token` → `TokenManager.refresh_token`)
   - Calls: `client.auth.refresh_session(refresh_token)` with retry/backoff.
   - On success:
     - Update `session['access_token']`, `session['refresh_token']`, `session['expires_at']`.
     - `save_callback(session)` → persists new session.
     - `set_client_callback(access_token, refresh_token)` → update client.
     - Optionally record metrics via `TokenRefreshMetrics.record_refresh(...)`.
   - On non-retryable error (invalid/expired refresh token):
     - `clear_callback()` → clear session from storage and memory.

### 4.2 Error-driven refresh

1. **Trigger**
   - A Supabase query raises an exception (e.g., 401, expired JWT).
   - Caller invokes: `cli.auth.handle_auth_error(error)`.

2. **Detection & handling** (`AuthErrorHandler`)
   - Checks error string and type for auth-related keywords.
   - If auth error:
     - Calls `Auth._refresh_token()` (same as proactive flow).
     - Returns `True` if refresh succeeded → caller retries operation once.

---

## 5. Logout Flow

1. **User action**
   - From main menu option 4: `cli.auth.logout()`.
   - Or when exiting via `exit_application(cli)` (see below).

2. **Auth.logout implementation**
   - If `self.session` is non-empty:
     - `client.auth.sign_out()` → invalidates tokens server-side.
     - `_clear_session_internal()` →
       - `clear_session(user_id=...)`:
         - **Production**: delete from `user_sessions` table.
         - **Development**: delete `~/.secure_share/session.json` / `.enc`.
       - `self.session = {}`.
   - If no session: logs that nothing is active.

Result:
- No local tokens, session cleared from storage, Supabase session revoked.

---

## 6. Application Exit Flow (with logout)

1. **Exit points**
   - Main menu option `0` (both authenticated and unauthenticated states).
   - KeyboardInterrupt / EOF in main menu.
   - KeyboardInterrupt / EOF at top-level `main.py` loop.

2. **Exit handler** (`cli/exit_handler.py`)
   - `exit_application(cli)` does:
     - If `cli` and `cli.auth` exist → `cli.auth.logout()` (best-effort).
     - Prints friendly exit message.
     - Raises `SystemExit(0)` to terminate the process.

3. **Integration**
   - `cli/main_menu.py` now always calls `exit_application(cli)` on:
     - Choice `0`.
     - KeyboardInterrupt / EOF while reading menu choice.
   - `main.py` top-level `try/except` for Ctrl+C / EOF also calls `exit_application(cli)`.

Result:
- On any clean exit path, the app logs the user out (if logged in), clears the session from storage, and exits with status code 0.
